In [9]:
# IMPORTS
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy import stats

In [10]:
# CONSTANTS
CLEAN_ROOT = "../../data/cmhc-rental-clean"
AVG_RENT_DIR = f"{CLEAN_ROOT}/avg-rent"
OUTPUT_DIR = f"{CLEAN_ROOT}/rental-increases"

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

PERIODS = [(2004, 2011), (2011, 2018), (2018, 2025), (2004, 2025), (2011, 2025)]
PERIOD_LABELS = [f"{s}_{e}" for s, e in PERIODS]

# Ontario rent control guidelines: maximum % increase allowed per year
RENT_CONTROL = {
    1991: 5.4, 1992: 6.0, 1993: 4.9, 1994: 3.2, 1995: 2.9,
    1996: 2.8, 1997: 2.8, 1998: 3.0, 1999: 3.0, 2000: 2.6,
    2001: 2.9, 2002: 3.9, 2003: 2.9, 2004: 2.9, 2005: 1.5,
    2006: 2.1, 2007: 2.6, 2008: 1.4, 2009: 1.8, 2010: 2.1,
    2011: 0.7, 2012: 3.1, 2013: 2.5, 2014: 0.8, 2015: 1.6,
    2016: 2.0, 2017: 1.5, 2018: 1.8, 2019: 1.8, 2020: 2.2,
    2021: 0.0, 2022: 1.2, 2023: 2.5, 2024: 2.5, 2025: 2.5,
}

In [11]:
# Load avg-rent data for all cities as {city_name -> pd.Series indexed by Year}
import glob, os

city_files = sorted(glob.glob(f"{AVG_RENT_DIR}/*.csv"))
cities = [os.path.splitext(os.path.basename(f))[0] for f in city_files]

city_series = {}
for path, city in tqdm(zip(city_files, cities), total=len(cities)):
    df = pd.read_csv(path)
    city_series[city] = pd.to_numeric(df.set_index("Year")["Total"], errors="coerce")

print(f"Loaded {len(city_series)} cities: {', '.join(cities)}")

100%|██████████| 34/34 [00:00<00:00, 489.19it/s]

Loaded 34 cities: ajax, barrie, brampton, brantford, burlington, cambridge, chatham-kent, clarington, greater sudbury, guelph, hamilton, kawartha lakes, kingston, kitchener, london, markham, milton, mississauga, newmarket, niagara falls, oakville, oshawa, ottawa, peterborough, pickering, richmond hill, sault ste. marie, st. catherines, thunder bay, toronto, vaughan, waterloo, whitby, windsor


In [12]:
def get_rent(series, year):
    """Return rent for a given year as float, or None if missing."""
    if year in series.index:
        val = series[year]
        return float(val) if pd.notna(val) else None
    return None


def allowed_rent_path(rent_start, start_year, end_year):
    """
    Compound rent_start through Ontario rent control guidelines for each year
    in [start_year, end_year]. Returns dict {year: max_allowed_rent}.
    The guideline for year Y caps the increase applied *in* year Y from Y-1.
    """
    path = {start_year: float(rent_start)}
    rent = float(rent_start)
    for year in range(start_year + 1, end_year + 1):
        rent *= 1 + RENT_CONTROL.get(year, 0.0) / 100
        path[year] = rent
    return path

## Compute Rent Increases

Five periods: **2004–2011 · 2011–2018 · 2018–2025 · 2004–2025 · 2011–2025**

**Raw** = actual change from period start to end.  
**Excess** = change above what Ontario rent control guidelines would have permitted (compounded annually).  
**pct** files: one column per period (5 cols). **total** files: extra dollars paid per month in the final year of the period vs the period start — one column per period (5 cols).

In [13]:
# pct_increase_raw  : (rent_end - rent_start) / rent_start * 100  for each period
# pct_increase_excess: above minus the % increase permitted under rent control

pct_raw_rows, pct_excess_rows = [], []

for city, series in tqdm(city_series.items()):
    raw_row = {"city": city}
    excess_row = {"city": city}
    for (s, e), label in zip(PERIODS, PERIOD_LABELS):
        rent_s = get_rent(series, s)
        rent_e = get_rent(series, e)
        if rent_s is not None and rent_e is not None and rent_s > 0:
            pct_raw = (rent_e - rent_s) / rent_s * 100
            raw_row[label] = round(pct_raw, 2)
            allowed = allowed_rent_path(rent_s, s, e)
            pct_allowed = (allowed[e] - rent_s) / rent_s * 100
            excess_row[label] = round(pct_raw - pct_allowed, 2)
        else:
            raw_row[label] = None
            excess_row[label] = None
    pct_raw_rows.append(raw_row)
    pct_excess_rows.append(excess_row)

pct_raw_df = pd.DataFrame(pct_raw_rows).set_index("city")
pct_excess_df = pd.DataFrame(pct_excess_rows).set_index("city")

pct_raw_df.to_csv(f"{OUTPUT_DIR}/pct_increase_raw.csv")
pct_excess_df.to_csv(f"{OUTPUT_DIR}/pct_increase_excess.csv")

print("Saved: pct_increase_raw.csv  |  pct_increase_excess.csv")
display(pct_raw_df.round(1))

100%|██████████| 34/34 [00:00<00:00, 6844.23it/s]

Saved: pct_increase_raw.csv  |  pct_increase_excess.csv


,2004_2011,2011_2018,2018_2025,2004_2025,2011_2025
city,,,,,
ajax,1.2,20.7,38.8,69.4,67.5
barrie,9.9,32.4,31.9,92.0,74.6
brampton,7.1,22.3,49.5,95.8,82.9
brantford,14.8,25.9,54.0,122.6,93.9
burlington,16.0,28.1,41.8,110.8,81.7
cambridge,23.6,29.0,71.0,172.7,120.6
chatham-kent,9.3,18.0,70.0,119.3,100.6
clarington,7.0,39.1,45.3,116.3,102.1
greater sudbury,34.5,20.7,50.2,143.8,81.2


In [14]:
# total_increase_raw   : extra dollars/month in the final year vs period start
# total_increase_excess: same, above what rent control would have permitted

total_raw_rows, total_excess_rows = [], []

for city, series in tqdm(city_series.items()):
    raw_row = {"city": city}
    excess_row = {"city": city}
    for (s, e), label in zip(PERIODS, PERIOD_LABELS):
        rent_s = get_rent(series, s)
        rent_e = get_rent(series, e)
        if rent_s is not None and rent_e is not None:
            allowed = allowed_rent_path(rent_s, s, e)
            raw_row[label] = round(rent_e - rent_s, 2)
            excess_row[label] = round(rent_e - allowed[e], 2)
        else:
            raw_row[label] = None
            excess_row[label] = None

    total_raw_rows.append(raw_row)
    total_excess_rows.append(excess_row)

total_raw_df = pd.DataFrame(total_raw_rows).set_index("city")
total_excess_df = pd.DataFrame(total_excess_rows).set_index("city")

total_raw_df.to_csv(f"{OUTPUT_DIR}/total_increase_raw.csv")
total_excess_df.to_csv(f"{OUTPUT_DIR}/total_increase_excess.csv")

print("Saved: total_increase_raw.csv  |  total_increase_excess.csv")
display(total_raw_df.round(0))

100%|██████████| 34/34 [00:00<00:00, 10162.94it/s]

Saved: total_increase_raw.csv  |  total_increase_excess.csv


,2004_2011,2011_2018,2018_2025,2004_2025,2011_2025
city,,,,,
ajax,12.0,215.0,486.0,713.0,701.0
barrie,88.0,315.0,411.0,814.0,726.0
brampton,70.0,236.0,641.0,947.0,877.0
brantford,103.0,207.0,543.0,853.0,750.0
burlington,150.0,306.0,583.0,1039.0,889.0
cambridge,161.0,244.0,771.0,1176.0,1015.0
chatham-kent,55.0,116.0,533.0,704.0,649.0
clarington,56.0,334.0,539.0,929.0,873.0
greater sudbury,210.0,169.0,495.0,874.0,664.0


In [15]:
# Convert all CSVs in OUTPUT_DIR to JSON (list of row dicts)
import json

for csv_path in sorted(glob.glob(f"{OUTPUT_DIR}/*.csv")):
    df = pd.read_csv(csv_path)
    json_path = csv_path.replace(".csv", ".json")
    with open(json_path, "w") as f:
        json.dump(df.to_dict(orient="records"), f)
    print(f"Saved {os.path.basename(json_path)}")

Saved pct_increase_excess.json
Saved pct_increase_raw.json
Saved total_increase_excess.json
Saved total_increase_raw.json
